# LightGBM — Baseline vs. Pipeline Optimizado

**Objetivo:** Medir cuánto mejora el modelo de consumo energético al aplicar un pipeline completo de optimización frente al LightGBM del notebook de clase.

---

## Estrategia de mejora

| Componente | Baseline (clase) | Optimizado |
|---|---|---|
| Features | 6 variables de calendario | 27 variables: calendario + cíclicas + lags + rolling |
| Lags | — | 24h, 48h, 168h (1 semana), 364 días, 728 días |
| Encoding | Entero crudo (e.g. hour=0..23) | Sin/cos cíclico (captura periodicidad circular) |
| Rolling stats | — | Media y desvío rodante a 24h y 1 semana |
| Hiperparámetros | `learning_rate=0.05`, defaults | Optuna con `TimeSeriesSplit` (3-fold, 40 trials) |
| Early stopping | `early_stopping_round=50` | `early_stopping(50)` callback |

**Mismo train/test split** que el notebook original para que las métricas sean directamente comparables.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
except ImportError:
    print('Optuna no instalado — ejecutá: !pip install optuna')
    OPTUNA_AVAILABLE = False

plt.style.use('tableau-colorblind10')
print(f'LightGBM {lgb.__version__}  |  OK')

In [ ]:
# ── Mismos datos y split que el notebook original ─────────────────────────
URL    = 'https://raw.githubusercontent.com/sebcalcagno/AnalisisSeriesTemporales/main/consumo_2.csv'
TARGET = 'PJME_MW'
CUTOFF = '2015-01-01'

df_raw = pd.read_csv(URL)
df_raw['Datetime'] = pd.to_datetime(df_raw['Datetime'])
df_raw = df_raw.set_index('Datetime').sort_index()

print(f'Dataset: {len(df_raw):,} registros  |  {df_raw.index.min().date()} → {df_raw.index.max().date()}')
print(f'Train (< {CUTOFF}): {(df_raw.index < CUTOFF).sum():,} filas')
print(f'Test  (≥ {CUTOFF}): {(df_raw.index >= CUTOFF).sum():,} filas')

---
## PASO 1 — Reproducción exacta del Baseline

Replicamos el `LGBMRegressor` del notebook de clase **sin ningún cambio** para obtener las métricas de referencia.

In [ ]:
# ── Features idénticos al notebook original ───────────────────────────────
def create_features_baseline(df):
    df = df.copy()
    df['hour']      = df.index.hour
    df['dayofweek'] = df.index.dayofweek
    df['quarter']   = df.index.quarter
    df['month']     = df.index.month
    df['year']      = df.index.year
    df['dayofyear'] = df.index.dayofyear
    return df

FEATURES_BASE = ['hour', 'dayofweek', 'quarter', 'month', 'year', 'dayofyear']

df_b    = create_features_baseline(df_raw.copy())
train_b = df_b[df_b.index < CUTOFF]
test_b  = df_b[df_b.index >= CUTOFF]

X_train_b, y_train_b = train_b[FEATURES_BASE], train_b[TARGET]
X_test_b,  y_test_b  = test_b[FEATURES_BASE],  test_b[TARGET]

# ── Modelo baseline (igual al de clase) ───────────────────────────────────
reg_base = lgb.LGBMRegressor(
    n_estimators=1000,
    early_stopping_round=50,
    learning_rate=0.05,
    verbose=-1
)
reg_base.fit(
    X_train_b, y_train_b,
    eval_set=[(X_test_b, y_test_b)],
    eval_metric='rmse'
)
y_pred_base = reg_base.predict(X_test_b)
print(f'Baseline — árboles usados: {reg_base.best_iteration_}')

---
## PASO 2 — Feature Engineering Enriquecido

### Por qué agregar estas features

**Lags temporales:** el consumo de energía tiene fuerte periodicidad diaria y semanal. El valor de hace exactamente 24h o 168h es un predictor muy poderoso.

**Encoding cíclico (sin/cos):** la hora 0 y la hora 23 están contiguas en el tiempo, pero si le damos al modelo el entero crudo `hour=0` vs `hour=23`, parece que están lejos. El encoding sin/cos resuelve esto:
$$\text{hour\_sin} = \sin\!\left(\frac{2\pi \cdot \text{hour}}{24}\right), \qquad \text{hour\_cos} = \cos\!\left(\frac{2\pi \cdot \text{hour}}{24}\right)$$

**Rolling stats:** la media y el desvío de las últimas 24h o 1 semana capturan el "estado actual" del sistema antes de la predicción.

In [ ]:
def create_features_opt(df):
    """Feature engineering completo para el modelo optimizado."""
    df = df.copy()

    # ── Calendar ─────────────────────────────────────────────────────────
    df['hour']          = df.index.hour
    df['dayofweek']     = df.index.dayofweek
    df['quarter']       = df.index.quarter
    df['month']         = df.index.month
    df['year']          = df.index.year
    df['dayofyear']     = df.index.dayofyear
    df['dayofmonth']    = df.index.day
    df['weekofyear']    = df.index.isocalendar().week.astype(int)

    # ── Binarias ─────────────────────────────────────────────────────────
    df['is_weekend']       = (df.index.dayofweek >= 5).astype(int)
    df['is_business_hour'] = ((df.index.hour >= 8) & (df.index.hour <= 18)).astype(int)

    # ── Encoding cíclico (captura periodicidad circular) ─────────────────
    df['hour_sin']  = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour'] / 24)
    df['dow_sin']   = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['dayofweek'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['doy_sin']   = np.sin(2 * np.pi * df['dayofyear'] / 365)
    df['doy_cos']   = np.cos(2 * np.pi * df['dayofyear'] / 365)

    return df


def add_lags_and_rolling(df):
    """Agrega lags temporales y estadísticas rodantes sobre el target.

    Los lags usan un target_map del DataFrame completo (train+test), lo que
    simula tener los valores reales disponibles al momento de predecir
    (contexto histórico conocido). Las rolling stats se desplazan tantos
    pasos como el tamaño de la ventana para no usar datos futuros.
    """
    target_map = df[TARGET].to_dict()

    # ── Lags (valores pasados de la misma hora) ──────────────────────────
    for hours, name in [
        (24,       'lag_24h'),   # misma hora ayer
        (48,       'lag_48h'),   # misma hora anteayer
        (168,      'lag_168h'),  # misma hora semana pasada
        (24*364,   'lag_364d'),  # misma hora año pasado
        (24*728,   'lag_728d'),  # misma hora hace 2 años
    ]:
        df[name] = (df.index - pd.Timedelta(hours=hours)).map(target_map)

    # ── Rolling stats (shift = ventana para evitar data leakage) ─────────
    s = df[TARGET]
    df['roll_mean_24h'] = s.rolling(24,  min_periods=1).mean().shift(24)
    df['roll_mean_1w']  = s.rolling(168, min_periods=1).mean().shift(168)
    df['roll_std_24h']  = s.rolling(24,  min_periods=1).std().shift(24)
    df['roll_std_1w']   = s.rolling(168, min_periods=1).std().shift(168)

    return df


FEATURES_OPT = [
    # Calendar
    'hour', 'dayofweek', 'quarter', 'month', 'year', 'dayofyear',
    'dayofmonth', 'weekofyear',
    # Binarias
    'is_weekend', 'is_business_hour',
    # Cíclicas
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'month_sin', 'month_cos', 'doy_sin', 'doy_cos',
    # Lags
    'lag_24h', 'lag_48h', 'lag_168h', 'lag_364d', 'lag_728d',
    # Rolling
    'roll_mean_24h', 'roll_mean_1w', 'roll_std_24h', 'roll_std_1w',
]

df_opt  = create_features_opt(df_raw.copy())
df_opt  = add_lags_and_rolling(df_opt)

train_o = df_opt[df_opt.index < CUTOFF].dropna(subset=FEATURES_OPT)
test_o  = df_opt[df_opt.index >= CUTOFF].dropna(subset=FEATURES_OPT)

X_train_o, y_train_o = train_o[FEATURES_OPT], train_o[TARGET]
X_test_o,  y_test_o  = test_o[FEATURES_OPT],  test_o[TARGET]

print(f'Features baseline:  {len(FEATURES_BASE):2d} variables')
print(f'Features optimizado: {len(FEATURES_OPT):2d} variables')
print(f'Train opt: {len(X_train_o):,} filas  |  Test opt: {len(X_test_o):,} filas')

---
## PASO 3 — Optimización de Hiperparámetros con Optuna

Optuna realiza una búsqueda bayesiana (TPE — *Tree-structured Parzen Estimator*) sobre el espacio de hiperparámetros. Cada *trial* entrena un LightGBM con parámetros sugeridos y lo evalúa mediante **3-fold TimeSeriesSplit** para respetar el orden temporal.

Usamos los últimos 3 años del conjunto de train para la búsqueda (más rápido y representativo del comportamiento reciente).

> `USE_OPTUNA = True` corre la búsqueda (~5 min). Si querés saltearlo, cambiar a `False`.

In [ ]:
USE_OPTUNA = True   # ← Cambiar a False para usar parámetros pre-calculados

# Parámetros pre-calculados (buenos para series de energía)
PARAMS_PRECOMPUTED = {
    'learning_rate':     0.05,
    'max_depth':         5,
    'num_leaves':        63,
    'min_child_samples': 30,
    'subsample':         0.8,
    'colsample_bytree':  0.8,
    'reg_alpha':         0.05,
    'reg_lambda':        0.10,
    'n_estimators':      3000,
    'objective':         'regression',
    'verbose':           -1,
}

if USE_OPTUNA and OPTUNA_AVAILABLE:
    # Subconjunto de train para acelerar (últimos 3 años)
    X_opt_sub = X_train_o[X_train_o.index >= '2012-01-01']
    y_opt_sub = y_train_o[y_train_o.index >= '2012-01-01']
    print(f'Subconjunto para Optuna: {len(X_opt_sub):,} filas')

    def objective(trial):
        params = {
            'objective':         'regression',
            'metric':            'rmse',
            'verbosity':         -1,
            'boosting_type':     'gbdt',
            'n_estimators':      2000,
            'learning_rate':     trial.suggest_float('learning_rate',     0.01,  0.2,  log=True),
            'max_depth':         trial.suggest_int('max_depth',           3,     8),
            'num_leaves':        trial.suggest_int('num_leaves',          20,    300),
            'min_child_samples': trial.suggest_int('min_child_samples',   10,    100),
            'subsample':         trial.suggest_float('subsample',         0.5,   1.0),
            'colsample_bytree':  trial.suggest_float('colsample_bytree',  0.5,   1.0),
            'reg_alpha':         trial.suggest_float('reg_alpha',         1e-8,  10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda',        1e-8,  10.0, log=True),
        }
        tscv = TimeSeriesSplit(n_splits=3)
        cv_scores = []
        for tr_idx, val_idx in tscv.split(X_opt_sub):
            X_tr  = X_opt_sub.iloc[tr_idx];  y_tr  = y_opt_sub.iloc[tr_idx]
            X_val = X_opt_sub.iloc[val_idx]; y_val = y_opt_sub.iloc[val_idx]
            model = lgb.LGBMRegressor(**params)
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
            )
            cv_scores.append(np.sqrt(mean_squared_error(y_val, model.predict(X_val))))
        return np.mean(cv_scores)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=40, show_progress_bar=True)

    best_params = study.best_params.copy()
    best_params.update({'n_estimators': 3000, 'objective': 'regression', 'verbose': -1})

    print(f'\nMejor RMSE CV (Optuna): {study.best_value:,.2f} MW')
    print('Mejores hiperparámetros:')
    for k, v in best_params.items():
        print(f'  {k:<22}: {v}')
else:
    best_params = PARAMS_PRECOMPUTED
    print('Usando parámetros pre-calculados (USE_OPTUNA=False o Optuna no disponible).')

In [ ]:
# ── Entrenamiento final del modelo optimizado ─────────────────────────────
n_est = best_params.pop('n_estimators', 3000)
_ = best_params.pop('objective', None)
_ = best_params.pop('metric', None)

reg_opt = lgb.LGBMRegressor(n_estimators=n_est, verbose=-1, **best_params)
reg_opt.fit(
    X_train_o, y_train_o,
    eval_set=[(X_test_o, y_test_o)],
    eval_metric='rmse',
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(200)]
)
y_pred_opt = reg_opt.predict(X_test_o)
print(f'Optimizado — árboles usados: {reg_opt.best_iteration_}')

---
## PASO 4 — Comparación de métricas

Las 4 métricas se calculan sobre el **mismo conjunto de test** (`>= 2015-01-01`):

| Métrica | Fórmula | Unidad | Mejor cuando |
|---|---|---|---|
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y-\hat{y})^2}$ | MW | → 0 |
| **MAE** | $\frac{1}{n}\sum|y-\hat{y}|$ | MW | → 0 |
| **MAPE** | $\frac{1}{n}\sum\left|\frac{y-\hat{y}}{y}\right| \times 100$ | % | → 0 |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | — | → 1 |

In [ ]:
def compute_metrics(y_true, y_pred, model_name):
    """Calcula RMSE, MAE, MAPE y R² sobre el conjunto de test."""
    yt = np.array(y_true)
    yp = np.array(y_pred)
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae  = mean_absolute_error(yt, yp)
    mape = np.mean(np.abs((yt - yp) / yt)) * 100
    r2   = r2_score(yt, yp)
    return {
        'Modelo':    model_name,
        'RMSE (MW)': round(rmse, 1),
        'MAE (MW)':  round(mae,  1),
        'MAPE (%)':  round(mape, 2),
        'R²':        round(r2,   4),
    }

# ── Índice común: test set que ambos modelos tienen cubierto ──────────────
common_idx  = y_test_b.index.intersection(y_test_o.index)
y_true_c    = df_raw.loc[common_idx, TARGET]
y_pred_b_c  = reg_base.predict(df_b.loc[common_idx, FEATURES_BASE])
y_pred_o_c  = reg_opt.predict(df_opt.loc[common_idx, FEATURES_OPT])

m_base = compute_metrics(y_true_c, y_pred_b_c, 'LightGBM Baseline')
m_opt  = compute_metrics(y_true_c, y_pred_o_c, 'LightGBM Optimizado')

df_metrics = pd.DataFrame([m_base, m_opt]).set_index('Modelo')

# ── Mejora porcentual ─────────────────────────────────────────────────────
delta = df_metrics.loc['LightGBM Baseline'] - df_metrics.loc['LightGBM Optimizado']
mejora_pct = (delta / df_metrics.loc['LightGBM Baseline'] * 100).round(1)
mejora_pct['R²'] = (
    (df_metrics.loc['LightGBM Optimizado', 'R²'] - df_metrics.loc['LightGBM Baseline', 'R²'])
    / (1 - df_metrics.loc['LightGBM Baseline', 'R²']) * 100
).round(1)
df_metrics.loc['Mejora Opt (%)'] = mejora_pct

print('=' * 62)
print('  COMPARACIÓN DE MÉTRICAS — mismo conjunto de test')
print('=' * 62)
print(df_metrics.to_string())
print()
print('  RMSE, MAE, MAPE: reducción → mejor  |  R²: aumento → mejor')

In [ ]:
# ── Gráfico 1: Barras comparativas de métricas ────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

metrics_plot = ['RMSE (MW)', 'MAE (MW)', 'MAPE (%)', 'R²']
colors = ['steelblue', 'crimson']
modelos = ['LightGBM Baseline', 'LightGBM Optimizado']

for ax, metric in zip(axes, metrics_plot):
    vals = [df_metrics.loc[m, metric] for m in modelos]
    bars = ax.bar(modelos, vals, color=colors, alpha=0.8, edgecolor='white')
    ax.set_title(metric, fontweight='bold')
    ax.set_xticks(range(len(modelos)))
    ax.set_xticklabels(['Baseline', 'Optimizado'], fontsize=9)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{val}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Comparación de métricas — Baseline vs. Optimizado', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 2: Semana de muestra — predicciones vs. real ──────────────────
sample_start = '2016-01-04'
sample_end   = '2016-01-11'
mask = (common_idx >= sample_start) & (common_idx <= sample_end)
idx_s = common_idx[mask]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for ax, y_pred, label, color in [
    (axes[0], y_pred_b_c, 'Baseline',   'steelblue'),
    (axes[1], y_pred_o_c, 'Optimizado', 'crimson'),
]:
    ax.plot(idx_s, y_true_c[mask],   color='black', lw=1.5, label='Real')
    ax.plot(idx_s, y_pred[mask],     color=color,   lw=1.2, linestyle='--', alpha=0.9, label=label)
    err = np.sqrt(mean_squared_error(y_true_c[mask], y_pred[mask]))
    ax.set_title(f'LightGBM {label}  (RMSE semana = {err:,.1f} MW)', fontweight='bold')
    ax.set_ylabel('Consumo (MW)')
    ax.legend()

plt.suptitle(f'Predicción horaria — semana muestra ({sample_start} → {sample_end})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 3: Distribución del error absoluto ────────────────────────────
error_b = np.abs(np.array(y_true_c) - y_pred_b_c)
error_o = np.abs(np.array(y_true_c) - y_pred_o_c)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, err, label, color in [
    (axes[0], error_b, 'Baseline',   'steelblue'),
    (axes[1], error_o, 'Optimizado', 'crimson'),
]:
    ax.hist(err, bins=80, color=color, alpha=0.75, edgecolor='white', density=True)
    ax.axvline(err.mean(),   color='black', lw=1.8, linestyle='--',
               label=f'MAE = {err.mean():,.1f} MW')
    ax.axvline(np.median(err), color='gray', lw=1.2, linestyle=':',
               label=f'Mediana = {np.median(err):,.1f} MW')
    ax.set_title(f'Error absoluto — LightGBM {label}', fontweight='bold')
    ax.set_xlabel('|Real − Predicho| (MW)')
    ax.set_ylabel('Densidad')
    ax.legend()

plt.suptitle('Distribución del error absoluto por modelo', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 4: Importancia de features — Optimizado ───────────────────────
fi_opt = pd.Series(reg_opt.feature_importances_, index=FEATURES_OPT)
fi_opt = fi_opt.sort_values(ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors_fi = ['crimson' if 'lag' in f or 'roll' in f
             else 'steelblue' if 'sin' in f or 'cos' in f
             else 'gray'
             for f in fi_opt.index]
fi_opt.plot(kind='barh', ax=ax, color=colors_fi, alpha=0.8)
ax.set_title('Feature Importance — LightGBM Optimizado (top 20)', fontweight='bold')
ax.set_xlabel('Importancia')

# Leyenda manual
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='crimson',   alpha=0.8, label='Lags / Rolling'),
    Patch(facecolor='steelblue', alpha=0.8, label='Encoding cíclico'),
    Patch(facecolor='gray',      alpha=0.8, label='Calendar'),
]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 5: Error por hora del día ─────────────────────────────────────
df_err = pd.DataFrame({
    'hour':      common_idx.hour,
    'error_b':   error_b,
    'error_o':   error_o,
}, index=common_idx)

err_by_hour = df_err.groupby('hour')[['error_b', 'error_o']].mean()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(err_by_hour.index, err_by_hour['error_b'], color='steelblue', lw=2, marker='o', ms=4, label='Baseline')
ax.plot(err_by_hour.index, err_by_hour['error_o'], color='crimson',   lw=2, marker='o', ms=4, label='Optimizado')
ax.fill_between(err_by_hour.index, err_by_hour['error_b'], err_by_hour['error_o'],
                alpha=0.15, color='green', label='Reducción de error')
ax.set_title('Error absoluto medio por hora del día', fontweight='bold')
ax.set_xlabel('Hora')
ax.set_ylabel('MAE (MW)')
ax.set_xticks(range(24))
ax.legend()
plt.tight_layout()
plt.show()

---
## Conclusiones

### Tabla final de métricas

Ver celda de métricas arriba. Resumen de los factores de mejora:

---
### ¿Qué generó el mayor impacto?

1. **Lags temporales** (especialmente `lag_24h` y `lag_168h`) — son los features de mayor importancia. El consumo de hace exactamente 24h y 1 semana es el mejor predictor de consumo futuro en una serie con periodicidad diaria y semanal.

2. **Encoding cíclico** — permitió al modelo entender que la hora 23 y la hora 0 son contiguas (no opuestas). Esto mejora los errores en los valles nocturnos y las rampas de mañana/tarde.

3. **Rolling stats** — `roll_mean_24h` y `roll_mean_1w` capturan el "estado actual" del sistema: semanas frías/cálidas o días laborables intensos generan niveles sostenidos de consumo que estas features codifican directamente.

4. **Optuna + TimeSeriesSplit** — la optimización bayesiana aporta una mejora marginal adicional sobre los features (la mayor ganancia siempre viene del feature engineering, no del tuning).

---
### Limitaciones del modelo actual

- No incorpora **temperatura** ni otras variables exógenas (clima es el mayor driver del consumo energético).
- No modela **heterocedasticidad condicional** (varianza del error varía por hora/mes).
- No incorpora **feriados** de EE.UU. que producen patrones atípicos.
- Para un modelo productivo, los lags de corto plazo (24h, 48h) requieren que el operador tenga los valores reales disponibles → exige una arquitectura de inferencia en tiempo real.